In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari

# ==========================================
# 0. PRE-PROCESSING UTILITIES
# ==========================================
def apply_db_cutoff(vol, cutoff_db=-5):
    v_abs = np.abs(vol)
    v_max = np.max(v_abs)
    if v_max == 0:
        return vol, 0.0
    thresh = v_max * (10**(cutoff_db / 20))
    v_thresh = np.where(v_abs >= thresh, vol, 0)
    sparsity = (np.count_nonzero(v_thresh == 0) / v_thresh.size) * 100
    return v_thresh.astype(np.float32), sparsity

# ==========================================
# 1. THE STITCHER ENGINE
# ==========================================
def run_stitcher_test(vol1, vol2, axis=2, grid=(60, 20), expected=0, tolerance=200, cutoff_db=-5):
    """ Finds shift using Filtered volumes. """
    v1, s1 = apply_db_cutoff(vol1, cutoff_db)
    v2, s2 = apply_db_cutoff(vol2, cutoff_db)
    
    ignore_top = 30
    z_dim, y_dim, x_dim = v1.shape
    tile_z = (z_dim - ignore_top) // grid[0]
    tile_y = y_dim // grid[1]
    
    all_shifts, all_weights = [], []
    
    for r in range(grid[0]):
        for c in range(grid[1]):
            zs, ze = ignore_top + (r * tile_z), ignore_top + ((r + 1) * tile_z)
            ys, ye = c * tile_y, (c + 1) * tile_y
            
            prof1 = np.max(np.abs(v1[zs:ze, ys:ye, :]), axis=(0, 1))
            prof2 = np.max(np.abs(v2[zs:ze, ys:ye, :]), axis=(0, 1))

            if np.std(prof1) < 1e-6 or np.max(prof1) == 0: continue

            p1_n = (prof1 - np.mean(prof1)) / (np.std(prof1) + 1e-10)
            p2_n = (prof2 - np.mean(prof2)) / (np.std(prof2) + 1e-10)
            
            corr = correlate(p1_n, p2_n, mode='full')
            lags = correlation_lags(len(p1_n), len(p2_n), mode='full')

            # Compute overlap length for each lag
            N = len(p1_n)
            M = len(p2_n)
            overlap = np.minimum(N, M + lags) - np.maximum(0, lags)
            overlap = np.maximum(overlap, 1)  # avoid division by zero

            # Normalize correlation by overlap
            corr = corr / overlap
            
            mask = (lags >= expected - tolerance) & (lags <= expected + tolerance)
            if not np.any(mask): continue
            corr[~mask] = -np.inf 

            all_shifts.append(lags[np.argmax(corr)])
            all_weights.append(np.max(corr))

    if not all_shifts:
        raise ValueError(f"No features survived the {cutoff_db}dB cutoff.")

    bins = np.arange(np.min(all_shifts), np.max(all_shifts) + 2) - 0.5
    counts, bin_edges = np.histogram(all_shifts, bins=bins, weights=all_weights)
    return int(bin_edges[np.argmax(counts)] + 0.5)

# ==========================================
# 2. CROSS-DIMENSIONALITY EXECUTION
# ==========================================
if __name__ == "__main__":
    DATA_ROOT = Path.cwd().parent / 'DATA' / '2D TFM Data'
    FILTERED_DIR = DATA_ROOT / 'FeC Smile 3MHz 04022026 Filtered'
    RAW_DIR = DATA_ROOT / 'FeC Smile 3MHz 04022026'
    
    try:
        # LOAD BOTH DATASETS
        v1_filt = np.load(FILTERED_DIR / "FeC_40_3_filtered_3D_TFM.npy")
        v2_filt = np.load(FILTERED_DIR / "FeC_40_4_filtered_3D_TFM.npy")
        
        v1_raw = np.load(RAW_DIR / "FeC_40_3_3D_TFM.npy")
        v2_raw = np.load(RAW_DIR / "FeC_40_4_3D_TFM.npy")
        
        # --- SHAPE CHECK ---
        if v1_filt.shape != v1_raw.shape:
            raise ValueError(
                f"DIMENSION MISMATCH DETECTED!\n"
                f"Filtered Data: {v1_filt.shape}\n"
                f"Raw Data:      {v1_raw.shape}\n"
                "The shift from filtered data cannot be applied to raw data of a different size."
            )
        print(f"[Check] Dimensions match: {v1_raw.shape}. Proceeding...")

    except FileNotFoundError as e:
        print(f"Data not found: {e}")
        exit()
    except ValueError as e:
        print(e)
        exit()

    # 1. Calculate shift using the "Low Dimensionality" (Filtered) volumes
    stitch_shift = run_stitcher_test(v1_filt, v2_filt, grid=(120, 40), cutoff_db=-10)
    print(f"\nCalculated Shift: {stitch_shift} px")

    # 2. Visualize using the "High Dimensionality" (Raw) volumes
    clim_raw = sorted([float(np.percentile(v1_raw, 0.1)), float(np.percentile(v1_raw, 99.9))])
    if clim_raw[0] == clim_raw[1]: clim_raw = [0, 1]

    viewer = napari.Viewer(title="Cross-Dimensionality Stitching (Validated)")

    # Display High-Res Vol 1
    viewer.add_image(v1_raw, name='Vol 1 (High-Res)', colormap='cyan', contrast_limits=clim_raw)
    
    # Display High-Res Vol 2 shifted
    trans = [0, 0, 0]; trans[2] = stitch_shift
    viewer.add_image(v2_raw, name=f'Vol 2 (High-Res Shifted)', 
                     colormap='magenta', blending='additive', translate=trans, contrast_limits=clim_raw)

    print(f"Viewing High-Dimensionality assembly. Shift applied: {stitch_shift} px.")
    napari.run()

[Check] Dimensions match: (400, 200, 200). Proceeding...

Calculated Shift: -122 px
Viewing High-Dimensionality assembly. Shift applied: -122 px.
